In [ ]:
import numpy as np
import pandas as pd
import sklearn
import torch
import scipy.stats

from tqdm import tqdm

from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib

In [ ]:
# PARAMETERS

NUM_RESAMPLES = 300
NUM_TRIALS = 100

MIN_SAMPLES_SPLIT = 10

SEED = 120013

FONT_SIZE = 20

SAVE = True
LOAD = False

rng = np.random.default_rng(seed=SEED)

# Breast Cancer

### Generate semi-synthetic data

In [ ]:
df_bc = pd.read_csv('./data/breastcancer_data.csv')
df_bc

In [ ]:
X_bc = df_bc.iloc[:,1:].to_numpy()
X_bc = np.concatenate([np.ones(shape=(X_bc.shape[0],1)),X_bc],axis=1)
y_bc = df_bc.iloc[:,0].to_numpy()

In [ ]:
if LOAD:
    w_true_bc = np.load('saved_data/w_true_bc_RF.npy')
else:
    LR = LogisticRegression()
    LR.fit(X_bc[:,1:], y_bc)
    w_true_bc = np.ones(shape=(X_bc.shape[1],1))
    w_true_bc[0] = LR.intercept_
    w_true_bc[1:] = LR.coef_.reshape(-1, 1)

In [ ]:
logit_true_bc = X_bc @ w_true_bc
probs_true_bc = expit(logit_true_bc)

### Initial RF on semi-synthetic data

In [ ]:
if LOAD:
    probs_pred_bc = np.load('saved_data/probs_pred_bc_RF.npy')
    y_obs_bc      = np.load('saved_data/y_obs_bc_RF.npy')

else:
    probs_pred_bc = np.zeros((X_bc.shape[0], NUM_TRIALS))
    y_obs_bc = np.zeros((X_bc.shape[0], NUM_TRIALS))
    
    for i in range(NUM_TRIALS):
        RF = RandomForestClassifier(min_samples_split=MIN_SAMPLES_SPLIT)
        y_obs_bc[:,i] = rng.binomial(1, probs_true_bc[:,0])
        RF.fit(X_bc, y_obs_bc[:,i])
    
        probs_pred_bc[:,i] = RF.predict_proba(X_bc)[:,1]

### Compute regret

In [ ]:
if LOAD:
    resamp_from_true_probs_bc = np.load('saved_data/resamp_from_true_probs_bc_RF.npy')
    resamp_from_pred_probs_bc = np.load('saved_data/resamp_from_pred_probs_bc_RF.npy')

else:
    resamp_from_true_probs_bc = np.zeros((X_bc.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    resamp_from_pred_probs_bc = np.zeros((X_bc.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    
    for i in tqdm(range(NUM_TRIALS)):
        
        for j in range(NUM_RESAMPLES):
            
            RF_from_true = RandomForestClassifier(min_samples_split=MIN_SAMPLES_SPLIT)
            RF_from_true.fit(X_bc, rng.binomial(1, probs_true_bc[:,0]))
                
            RF_from_pred = RandomForestClassifier(min_samples_split=MIN_SAMPLES_SPLIT)
            RF_from_pred.fit(X_bc, rng.binomial(1, probs_pred_bc[:,i]))
    
            resamp_from_true_probs_bc[:,i,j] = RF_from_true.predict_proba(X_bc)[:,1]
            resamp_from_pred_probs_bc[:,i,j] = RF_from_pred.predict_proba(X_bc)[:,1]

In [ ]:
regret_probs_true_bc = np.var(resamp_from_true_probs_bc, axis=2, ddof=1)
regret_probs_pred_bc = np.var(resamp_from_pred_probs_bc, axis=2, ddof=1)

# Bank

### Generate semi-synthetic data

In [ ]:
df_bk = pd.read_csv('./data/bank_data.csv')
df_bk

In [ ]:
X_bk = df_bk.iloc[:,1:].to_numpy()
X_bk = np.concatenate([np.ones(shape=(X_bk.shape[0],1)),X_bk],axis=1)
y_bk = df_bk.iloc[:,0].to_numpy()

In [ ]:
_, R = np.linalg.qr(X_bk)
X_bk = X_bk[:,np.abs(np.diag(R)) > 1e-10]

In [ ]:
if LOAD:
    w_true_bk = np.load('saved_data/w_true_bk_RF.npy')
else:
    LR = LogisticRegression()
    LR.fit(X_bk[:,1:], y_bk)
    w_true_bk = np.ones(shape=(X_bk.shape[1],1))
    w_true_bk[0] = LR.intercept_
    w_true_bk[1:] = LR.coef_.reshape(-1, 1)

In [ ]:
logit_true_bk = X_bk @ w_true_bk
probs_true_bk = expit(logit_true_bk)
om_probs_true_bk = expit(-logit_true_bk)

### Initial RF on semi-synthetic data

In [ ]:
if LOAD:
    probs_pred_bk = np.load('saved_data/probs_pred_bk_RF.npy')
    y_obs_bk      = np.load('saved_data/y_obs_bk_RF.npy')

else:
    probs_pred_bk = np.zeros((X_bk.shape[0], NUM_TRIALS))
    y_obs_bk = np.zeros((X_bk.shape[0], NUM_TRIALS))
    
    for i in range(NUM_TRIALS):
        RF = RandomForestClassifier(min_samples_split=MIN_SAMPLES_SPLIT)
        y_obs_bk[:,i] = rng.binomial(1, probs_true_bk[:,0])
        RF.fit(X_bk, y_obs_bk[:,i])
    
        probs_pred_bk[:,i] = RF.predict_proba(X_bk)[:,1]

### Compute regret

In [ ]:
if LOAD:
    resamp_from_true_probs_bk = np.load('saved_data/resamp_from_true_probs_bk_RF.npy')
    resamp_from_pred_probs_bk = np.load('saved_data/resamp_from_pred_probs_bk_RF.npy')

else:
    resamp_from_true_probs_bk = np.zeros((X_bk.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    resamp_from_pred_probs_bk = np.zeros((X_bk.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    
    for i in tqdm(range(NUM_TRIALS)):
        
        for j in range(NUM_RESAMPLES):
            
            RF_from_true = RandomForestClassifier(min_samples_split=MIN_SAMPLES_SPLIT)
            RF_from_true.fit(X_bk, rng.binomial(1, probs_true_bk[:,0]))
                
            RF_from_pred = RandomForestClassifier(min_samples_split=MIN_SAMPLES_SPLIT)
            RF_from_pred.fit(X_bk, rng.binomial(1, probs_pred_bk[:,i]))
    
            resamp_from_true_probs_bk[:,i,j] = RF_from_true.predict_proba(X_bk)[:,1]
            resamp_from_pred_probs_bk[:,i,j] = RF_from_pred.predict_proba(X_bk)[:,1]

In [ ]:
regret_probs_true_bk = np.var(resamp_from_true_probs_bk, axis=2, ddof=1)
regret_probs_pred_bk = np.var(resamp_from_pred_probs_bk, axis=2, ddof=1)

# Apnea

### Generate semi-synthetic data

In [ ]:
df_ap = pd.read_csv('./data/apnea/apnea_data.csv')
df_ap

In [ ]:
df_ap = df_ap.drop(columns=['Age','Sex','bmi_lt_25']).join(pd.get_dummies(df_ap.loc[:,['Age','Sex']], drop_first=True, dtype=int))
df_ap

In [ ]:
X_ap = df_ap.iloc[:,1:].to_numpy()
X_ap = np.concatenate([np.ones(shape=(X_ap.shape[0],1)),X_ap],axis=1)
y_ap = df_ap.iloc[:,0].to_numpy()

In [ ]:
if LOAD:
    w_true_ap = np.load('saved_data/w_true_ap_RF.npy')
else:
    LR = LogisticRegression()
    LR.fit(X_ap[:,1:], y_ap)
    w_true_ap = np.ones(shape=(X_ap.shape[1],1))
    w_true_ap[0] = LR.intercept_
    w_true_ap[1:] = LR.coef_.reshape(-1, 1)

In [ ]:
logit_true_ap = X_ap @ w_true_ap
probs_true_ap = expit(logit_true_ap)
om_probs_true_ap = expit(-logit_true_ap)

### Initial RF on semi-synthetic data

In [ ]:
if LOAD:
    probs_pred_ap = np.load('saved_data/probs_pred_ap_RF.npy')
    y_obs_ap      = np.load('saved_data/y_obs_ap_RF.npy')

else:
    probs_pred_ap = np.zeros((X_ap.shape[0], NUM_TRIALS))
    y_obs_ap = np.zeros((X_ap.shape[0], NUM_TRIALS))
    
    for i in range(NUM_TRIALS):
        RF = RandomForestClassifier(min_samples_split=MIN_SAMPLES_SPLIT)
        y_obs_ap[:,i] = rng.binomial(1, probs_true_ap[:,0])
        RF.fit(X_ap, y_obs_ap[:,i])
    
        probs_pred_ap[:,i] = RF.predict_proba(X_ap)[:,1]

### Compute regret

In [ ]:
if LOAD:
    resamp_from_true_probs_ap = np.load('saved_data/resamp_from_true_probs_ap_RF.npy')
    resamp_from_pred_probs_ap = np.load('saved_data/resamp_from_pred_probs_ap_RF.npy')

else:
    resamp_from_true_probs_ap = np.zeros((X_ap.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    resamp_from_pred_probs_ap = np.zeros((X_ap.shape[0], NUM_TRIALS, NUM_RESAMPLES))
    
    for i in tqdm(range(NUM_TRIALS)):
        
        for j in range(NUM_RESAMPLES):
            
            RF_from_true = RandomForestClassifier(min_samples_split=MIN_SAMPLES_SPLIT)
            RF_from_true.fit(X_ap, rng.binomial(1, probs_true_ap[:,0]))
                
            RF_from_pred = RandomForestClassifier(min_samples_split=MIN_SAMPLES_SPLIT)
            RF_from_pred.fit(X_ap, rng.binomial(1, probs_pred_ap[:,i]))
    
            resamp_from_true_probs_ap[:,i,j] = RF_from_true.predict_proba(X_ap)[:,1]
            resamp_from_pred_probs_ap[:,i,j] = RF_from_pred.predict_proba(X_ap)[:,1]

In [ ]:
regret_probs_true_ap = np.var(resamp_from_true_probs_ap, axis=2, ddof=1)
regret_probs_pred_ap = np.var(resamp_from_pred_probs_ap, axis=2, ddof=1)

### Save data

In [ ]:
if SAVE:
    np.save('saved_data/w_true_bc_RF.npy', w_true_bc)
    np.save('saved_data/probs_pred_bc_RF.npy', probs_pred_bc)
    np.save('saved_data/y_obs_bc_RF.npy', y_obs_bc)
    np.save('saved_data/resamp_from_true_probs_bc_RF.npy', resamp_from_true_probs_bc)
    np.save('saved_data/resamp_from_pred_probs_bc_RF.npy', resamp_from_pred_probs_bc)

    np.save('saved_data/w_true_bk_RF.npy', w_true_bk)
    np.save('saved_data/probs_pred_bk_RF.npy', probs_pred_bk)
    np.save('saved_data/y_obs_bk_RF.npy', y_obs_bk)
    np.save('saved_data/resamp_from_true_probs_bk_RF.npy', resamp_from_true_probs_bk)
    np.save('saved_data/resamp_from_pred_probs_bk_RF.npy', resamp_from_pred_probs_bk)

    np.save('saved_data/w_true_ap_RF.npy', w_true_ap)
    np.save('saved_data/probs_pred_ap_RF.npy', probs_pred_ap)
    np.save('saved_data/y_obs_ap_RF.npy', y_obs_ap)
    np.save('saved_data/resamp_from_true_probs_ap_RF.npy', resamp_from_true_probs_ap)
    np.save('saved_data/resamp_from_pred_probs_ap_RF.npy', resamp_from_pred_probs_ap)

# Plots

### Single trial

In [ ]:
matplotlib.rcParams['savefig.dpi'] = 300

plt.rcParams.update({'font.size': FONT_SIZE})

plt.errorbar(regret_probs_true_bc[:,0],
             np.median(regret_probs_pred_bc, axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_bc, [0,1],axis=1) 
                        -np.median(regret_probs_pred_bc, axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.9,0.9,1), zorder=0, rasterized=True)
plt.errorbar(regret_probs_true_bc[:,0],
             np.median(regret_probs_pred_bc, axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_bc, [0.25,0.75],axis=1) 
                        -np.median(regret_probs_pred_bc, axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.5,.5,1), zorder=1, rasterized=True)
plt.ylabel("Regret (estimated)")
plt.xlabel("Regret (true)")

xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
plt.plot([0,1],[0,1],'k--',zorder=2)
plt.xlim(xlim)
plt.ylim(ylim)

plt.gca().set_box_aspect(3/4)
plt.savefig('plots/regret-bc-RF.pdf', bbox_inches='tight')
plt.show()

plt.errorbar(regret_probs_true_bk[:,0],
             np.median(regret_probs_pred_bk, axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_bk, [0,1],axis=1) 
                        -np.median(regret_probs_pred_bk, axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.9,0.9,1), zorder=0, rasterized=True)
plt.errorbar(regret_probs_true_bk[:,0],
             np.median(regret_probs_pred_bk, axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_bk, [0.25,0.75],axis=1) 
                        -np.median(regret_probs_pred_bk, axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.5,0.5,1), zorder=0, rasterized=True)
plt.ylabel("Regret (estimated)")
plt.xlabel("Regret (true)")

xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
plt.plot([0,1],[0,1],'k--',zorder=1)
plt.xlim(xlim)
plt.ylim(ylim)

plt.gca().set_box_aspect(3/4)
plt.savefig('plots/regret-bk-RF.pdf', bbox_inches='tight')
plt.show()

plt.errorbar(regret_probs_true_ap[:,0],
             np.median(regret_probs_pred_ap, axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_ap, [0,1],axis=1) 
                        -np.median(regret_probs_pred_ap, axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.9,0.9,1), zorder=0)
plt.errorbar(regret_probs_true_ap[:,0],
             np.median(regret_probs_pred_ap, axis=1),
             yerr=np.abs(np.quantile(regret_probs_pred_ap, [0.25,0.75],axis=1) 
                        -np.median(regret_probs_pred_ap, axis=1).reshape(1,-1)),
             fmt='b.', ecolor=(0.5,0.5,1), zorder=0)
plt.ylabel("Regret (estimated)")
plt.xlabel("Regret (true)")

xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
plt.plot([0,1],[0,1],'k--',zorder=1)
plt.xlim(xlim)
plt.ylim(ylim)

plt.gca().set_box_aspect(3/4)
plt.savefig('plots/regret-ap-RF.pdf', bbox_inches='tight')
plt.show()